# Giải thích model churn cho đội retention

Bạn đã có model churn trên bộ Telco (7 043 khách). Giám đốc retention hỏi: "Model dựa vào gì? Với khách
này, vì sao điểm rủi ro cao? Chúng ta nên làm gì khác?" Nhiệm vụ: trả lời bằng permutation importance và
SHAP, chỉ ra chỗ hai cách giải thích có thể đánh lừa, và viết `REPORT.md`.

**Cách làm bài**
- Hai hàm chấm điểm: `permutation_importance_df` và `group_shap`. Phần còn lại là phân tích tự do trong các
  cell `explore` (test bỏ qua).
- Chấm: `uv run pytest tracks/data-science/modules/11-classic-ml/exercises/advanced/01-churn-explain -q`

In [ ]:
import numpy as np
import pandas as pd

## 1. Permutation importance tự cài

`permutation_importance_df(model, X, y, score_fn, n_repeats=5, seed=0)`:

- `base = score_fn(y, model.predict_proba(X)[:, 1])` (score càng cao càng tốt, ví dụ ROC-AUC).
- Với mỗi cột của `X` (DataFrame **thô**, trước one-hot, để một biến phân loại được hoán vị nguyên khối) và
  mỗi lần lặp: hoán vị cột đó (bản sao, không sửa `X`), tính lại score; importance là `base − score`.
- Dùng một `np.random.default_rng(seed)` cho cả lần gọi.
- Trả DataFrame index là tên cột, cột `importance_mean`, `importance_std`, sắp `importance_mean` giảm dần.

In [ ]:
def permutation_importance_df(model, X: pd.DataFrame, y, score_fn, n_repeats: int = 5,
                              seed: int = 0) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

## 2. Gộp SHAP về biến gốc

SHAP tính trên ma trận đã one-hot, nên `Contract` thành ba cột `cat__Contract_Month-to-month`,
`cat__Contract_One year`, `cat__Contract_Two year`. Nhờ tính cộng (additivity), SHAP của biến gốc là **tổng**
SHAP các cột con.

`group_shap(values, original_features)`: `values` là DataFrame (một dòng mỗi khách, cột là tên sau
`ColumnTransformer.get_feature_names_out()`, dạng `<transformer>__<tên>`). Bỏ tiền tố tới `__`; cột thuộc về
biến gốc **dài nhất** mà phần còn lại bằng đúng tên đó hoặc bắt đầu bằng `tên + "_"`. Trả DataFrame cột theo
thứ tự `original_features`.

In [ ]:
def group_shap(values: pd.DataFrame, original_features: list[str]) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: dữ liệu và model (tải bằng: uv run python data/download_dataset.py telco-churn)
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
df = pd.read_csv(ROOT / "data/raw/telco-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df.TotalCharges, errors="coerce").fillna(0.0)
y = (df.Churn == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])
NUM = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
CAT = [c for c in X.columns if c not in NUM]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

pre = ColumnTransformer([("num", "passthrough", NUM),
                         ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), CAT)])
model = Pipeline([("pre", pre), ("clf", RandomForestClassifier(300, min_samples_leaf=10, random_state=0,
                                                              n_jobs=-1))]).fit(X_tr, y_tr)
print("ROC-AUC test:", round(roc_auc_score(y_te, model.predict_proba(X_te)[:, 1]), 3))

## 3. Phân tích (tự do)

Gợi ý thứ tự, mỗi bước một hoặc vài cell `explore`:

1. Permutation importance trên **test** bằng hàm của bạn. So với `feature_importances_` (MDI) của random
   forest: thứ hạng khác nhau ở đâu, vì sao?
2. `MonthlyCharges`, `TotalCharges`, `tenure` tương quan mạnh (`TotalCharges ≈ tenure × MonthlyCharges`).
   Điều đó ảnh hưởng permutation importance thế nào? Thử bỏ một cột và train lại.
3. SHAP: `shap.TreeExplainer(model[-1])` trên `model[0].transform(X_te)` (đổi sang DataFrame với
   `get_feature_names_out()`), lấy SHAP của lớp 1. Gộp bằng `group_shap`, xếp theo trung bình |SHAP|.
4. Giải thích 2–3 khách cụ thể (một rủi ro cao, một rủi ro thấp) bằng SHAP của từng người.
5. Một câu "nếu… thì…" mà đội retention muốn nghe (ví dụ "chuyển khách sang hợp đồng 1 năm thì giảm churn
   bao nhiêu"): SHAP có trả lời được không? Viết rõ trong REPORT.

In [ ]:
# explore: TODO bước 1

In [ ]:
# explore: TODO bước 2

In [ ]:
# explore: TODO bước 3

In [ ]:
# explore: TODO bước 4

## 4. Báo cáo

Viết `REPORT.md` (cùng thư mục với notebook) cho Giám đốc retention theo khung có sẵn.